In [2]:
"""
Professional Animated Feature Distribution Analysis
Save as: src/visualization/feature_distribution_animated.py
Run: python src/visualization/feature_distribution_animated.py

Features:
- Interactive distribution plots with KDE
- Real-time statistical summaries
- Animated correlation matrix heatmap
- 3D feature interaction visualization
- Box plots with violin overlays
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create directories
VIZ_PATH = Path("visualizations/features")
VIZ_PATH.mkdir(parents=True, exist_ok=True)
HTML_PATH = VIZ_PATH / 'html'
HTML_PATH.mkdir(parents=True, exist_ok=True)

# Professional color palette
COLORS = {
    'primary': '#1E88E5',
    'secondary': '#DC143C',
    'success': '#2ECC71',
    'warning': '#F39C12',
    'info': '#00ACC1',
    'dark': '#2C3E50',
    'light': '#ECF0F1',
    'purple': '#9B59B6',
    'orange': '#E67E22'
}


def plot_animated_feature_distributions(df, features=None, n_cols=3, save_path=None):
    """
    Create interactive feature distribution dashboard
    
    Parameters:
    - df: DataFrame with features
    - features: list of feature names (default: first 12)
    - n_cols: number of columns in subplot grid
    - save_path: path to save the plot
    """
    print("\n📊 Creating animated feature distribution dashboard...")
    
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns[:12]
    
    features = list(features)[:12]
    n_features = len(features)
    n_rows = (n_features + n_cols - 1) // n_cols
    
    # Create subplot grid
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=features,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    # Calculate statistics for each feature
    stats_dict = {}
    
    for i, feature in enumerate(features):
        row = i // n_cols + 1
        col = i % n_cols + 1
        
        data = df[feature].dropna()
        mean_val = data.mean()
        median_val = data.median()
        skewness = stats.skew(data)
        kurtosis = stats.kurtosis(data)
        
        stats_dict[feature] = {
            'mean': mean_val,
            'median': median_val,
            'skewness': skewness,
            'kurtosis': kurtosis
        }
        
        # Add histogram with KDE
        fig.add_trace(
            go.Histogram(
                x=data,
                nbinsx=50,
                name=feature,
                histnorm='probability density',
                marker_color=COLORS['primary'],
                opacity=0.7,
                hovertemplate='<b>%{x:.4f}</b><br>Density: %{y:.4f}<extra></extra>'
            ),
            row=row, col=col
        )
        
        # Add KDE line
        kde = stats.gaussian_kde(data)
        x_range = np.linspace(data.min(), data.max(), 200)
        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=kde(x_range),
                mode='lines',
                name=f'{feature} KDE',
                line=dict(color=COLORS['secondary'], width=2),
                showlegend=False,
                hovertemplate='<b>%{x:.4f}</b><br>Density: %{y:.4f}<extra></extra>'
            ),
            row=row, col=col
        )
        
        # Add mean line
        fig.add_vline(
            x=mean_val, line_dash="dash", line_color="red",
            annotation_text=f'μ={mean_val:.4f}',
            annotation_position="top",
            row=row, col=col
        )
        
        # Add median line
        fig.add_vline(
            x=median_val, line_dash="dash", line_color="green",
            annotation_text=f'med={median_val:.4f}',
            annotation_position="bottom",
            row=row, col=col
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='<b>Feature Distributions Analysis</b><br><sub>Interactive histograms with KDE and summary statistics</sub>',
            x=0.5,
            font=dict(size=18, family='Arial Black')
        ),
        height=300 * n_rows,
        width=400 * n_cols,
        showlegend=False,
        template='plotly_white',
        hovermode='closest'
    )
    
    # Update axes labels
    for i, feature in enumerate(features):
        row = i // n_cols + 1
        col = i % n_cols + 1
        fig.update_xaxes(title_text=feature, row=row, col=col)
        fig.update_yaxes(title_text='Density', row=row, col=col)
    
    # Save files
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved feature distributions to {save_path}")
    
    fig.write_html(HTML_PATH / 'feature_distributions.html')
    fig.write_image(VIZ_PATH / 'feature_distributions.png', 
                    width=400 * n_cols, height=300 * n_rows)
    
    # Print statistics summary
    print("\n📊 Feature Statistics Summary:")
    for feature, stats_data in stats_dict.items():
        print(f"   {feature}: μ={stats_data['mean']:.4f}, "
              f"med={stats_data['median']:.4f}, "
              f"skew={stats_data['skewness']:.3f}, "
              f"kurt={stats_data['kurtosis']:.3f}")
    
    return fig


def plot_animated_target_distribution(y_train, y_test, save_path=None):
    """
    Create interactive target distribution dashboard
    
    Parameters:
    - y_train: training target values
    - y_test: test target values
    - save_path: path to save the plot
    """
    print("\n📊 Creating animated target distribution dashboard...")
    
    # Calculate statistics
    train_stats = {
        'mean': np.mean(y_train),
        'std': np.std(y_train),
        'skew': stats.skew(y_train),
        'kurt': stats.kurtosis(y_train)
    }
    
    test_stats = {
        'mean': np.mean(y_test),
        'std': np.std(y_test),
        'skew': stats.skew(y_test),
        'kurt': stats.kurtosis(y_test)
    }
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Target Distribution (Train vs Test)', 
                       'Box Plot Comparison',
                       'Q-Q Plot (Train)', 
                       'Q-Q Plot (Test)'),
        vertical_spacing=0.12,
        horizontal_spacing=0.12,
        specs=[[{'type': 'scatter'}, {'type': 'box'}],
               [{'type': 'scatter'}, {'type': 'scatter'}]]
    )
    
    # 1. Histogram with KDE
    fig.add_trace(
        go.Histogram(
            x=y_train, nbinsx=50, name='Train',
            marker_color=COLORS['primary'], opacity=0.6,
            histnorm='probability density',
            hovertemplate='<b>Train:</b> %{x:.4f}<br>Density: %{y:.4f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Histogram(
            x=y_test, nbinsx=50, name='Test',
            marker_color=COLORS['secondary'], opacity=0.6,
            histnorm='probability density',
            hovertemplate='<b>Test:</b> %{x:.4f}<br>Density: %{y:.4f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add KDE lines
    kde_train = stats.gaussian_kde(y_train)
    kde_test = stats.gaussian_kde(y_test)
    x_range = np.linspace(min(y_train.min(), y_test.min()), 
                          max(y_train.max(), y_test.max()), 200)
    
    fig.add_trace(
        go.Scatter(
            x=x_range, y=kde_train(x_range),
            mode='lines', name='Train KDE',
            line=dict(color=COLORS['primary'], width=2),
            showlegend=False,
            hovertemplate='Train KDE: %{y:.4f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=x_range, y=kde_test(x_range),
            mode='lines', name='Test KDE',
            line=dict(color=COLORS['secondary'], width=2, dash='dash'),
            showlegend=False,
            hovertemplate='Test KDE: %{y:.4f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. Box plot
    fig.add_trace(
        go.Box(
            y=y_train, name='Train',
            marker_color=COLORS['primary'],
            boxmean='sd',
            hovertemplate='Train<br>Median: %{median:.4f}<br>Mean: %{mean:.4f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Box(
            y=y_test, name='Test',
            marker_color=COLORS['secondary'],
            boxmean='sd',
            hovertemplate='Test<br>Median: %{median:.4f}<br>Mean: %{mean:.4f}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # 3. Q-Q plot for train
    sorted_train = np.sort(y_train)
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_train)))
    
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles, y=sorted_train,
            mode='markers', name='Train Q-Q',
            marker=dict(size=4, color=COLORS['primary'], opacity=0.6),
            hovertemplate='<b>Theoretical:</b> %{x:.4f}<br><b>Sample:</b> %{y:.4f}<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Reference line for train
    slope, intercept = np.polyfit(theoretical_quantiles, sorted_train, 1)
    line_x = np.array([theoretical_quantiles.min(), theoretical_quantiles.max()])
    line_y = slope * line_x + intercept
    fig.add_trace(
        go.Scatter(
            x=line_x, y=line_y, mode='lines',
            name='Reference',
            line=dict(color='red', width=2, dash='dash'),
            showlegend=False
        ),
        row=2, col=1
    )
    
    # 4. Q-Q plot for test
    sorted_test = np.sort(y_test)
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles, y=sorted_test,
            mode='markers', name='Test Q-Q',
            marker=dict(size=4, color=COLORS['secondary'], opacity=0.6),
            hovertemplate='<b>Theoretical:</b> %{x:.4f}<br><b>Sample:</b> %{y:.4f}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Reference line for test
    slope, intercept = np.polyfit(theoretical_quantiles, sorted_test, 1)
    line_y = slope * line_x + intercept
    fig.add_trace(
        go.Scatter(
            x=line_x, y=line_y, mode='lines',
            name='Reference',
            line=dict(color='red', width=2, dash='dash'),
            showlegend=False
        ),
        row=2, col=2
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="solid", line_color="gray", opacity=0.5, row=1, col=1)
    
    fig.update_layout(
        title=dict(
            text='<b>Target Variable Analysis</b><br><sub>Distribution comparison and normality testing</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        height=800,
        width=1100,
        showlegend=True,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    
    fig.update_xaxes(title_text='Value', row=1, col=1)
    fig.update_yaxes(title_text='Density', row=1, col=1)
    fig.update_yaxes(title_text='Target Return', row=1, col=2)
    fig.update_xaxes(title_text='Theoretical Quantiles', row=2, col=1)
    fig.update_yaxes(title_text='Sample Quantiles', row=2, col=1)
    fig.update_xaxes(title_text='Theoretical Quantiles', row=2, col=2)
    fig.update_yaxes(title_text='Sample Quantiles', row=2, col=2)
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved target distribution to {save_path}")
    
    fig.write_html(HTML_PATH / 'target_distribution.html')
    fig.write_image(VIZ_PATH / 'target_distribution.png', width=1100, height=800)
    
    # Print statistics
    print("\n📊 Target Statistics:")
    print(f"   Train: μ={train_stats['mean']:.4f}, σ={train_stats['std']:.4f}, "
          f"skew={train_stats['skew']:.3f}, kurt={train_stats['kurt']:.3f}")
    print(f"   Test:  μ={test_stats['mean']:.4f}, σ={test_stats['std']:.4f}, "
          f"skew={test_stats['skew']:.3f}, kurt={test_stats['kurt']:.3f}")
    
    return fig


def plot_animated_correlation_matrix(df, features=None, figsize=(1000, 800), save_path=None):
    """
    Create interactive correlation matrix heatmap
    
    Parameters:
    - df: DataFrame with features
    - features: list of feature names (default: first 20)
    - figsize: figure size (width, height)
    - save_path: path to save the plot
    """
    print("\n📊 Creating animated correlation matrix...")
    
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns[:20]
    
    features = list(features)
    corr_matrix = df[features].corr()
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=features,
        y=features,
        colorscale='RdBu',
        zmin=-1,
        zmax=1,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        hovertemplate='<b>%{x}</b> vs <b>%{y}</b><br>Correlation: %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text='<b>Feature Correlation Matrix</b><br><sub>Interactive heatmap of feature relationships</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        width=figsize[0],
        height=figsize[1],
        xaxis=dict(tickangle=45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9)),
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved correlation matrix to {save_path}")
    
    fig.write_html(HTML_PATH / 'correlation_matrix.html')
    fig.write_image(VIZ_PATH / 'correlation_matrix.png', width=figsize[0], height=figsize[1])
    
    # Find high correlations
    print("\n📊 High Correlation Pairs (>0.8 or <-0.8):")
    for i in range(len(features)):
        for j in range(i+1, len(features)):
            corr = corr_matrix.iloc[i, j]
            if abs(corr) > 0.8:
                print(f"   {features[i]} vs {features[j]}: {corr:.3f}")
    
    return fig


def plot_animated_violin_distributions(df, features=None, save_path=None):
    """
    Create interactive violin plot distributions
    
    Parameters:
    - df: DataFrame with features
    - features: list of feature names (default: first 8)
    - save_path: path to save the plot
    """
    print("\n📊 Creating animated violin distribution dashboard...")
    
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns[:8]
    
    features = list(features)
    
    # Prepare data for violin plot
    plot_data = []
    for feature in features:
        plot_data.extend([{'Feature': feature, 'Value': val} for val in df[feature].dropna()])
    
    plot_df = pd.DataFrame(plot_data)
    
    fig = px.violin(plot_df, x='Feature', y='Value', box=True, points='all',
                    title='<b>Feature Distribution Violin Plot</b><br><sub>Box plot with kernel density estimation</sub>',
                    color='Feature',
                    color_discrete_sequence=px.colors.qualitative.Set2)
    
    fig.update_layout(
        height=500,
        width=1000,
        showlegend=False,
        template='plotly_white',
        xaxis_title='Features',
        yaxis_title='Value'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved violin distributions to {save_path}")
    
    fig.write_html(HTML_PATH / 'violin_distributions.html')
    fig.write_image(VIZ_PATH / 'violin_distributions.png', width=1000, height=500)
    
    return fig


def plot_3d_feature_interaction(df, feature1, feature2, feature3, save_path=None):
    """
    Create 3D scatter plot for feature interaction analysis
    
    Parameters:
    - df: DataFrame with features
    - feature1: first feature name (x-axis)
    - feature2: second feature name (y-axis)
    - feature3: third feature name (z-axis)
    - save_path: path to save the plot
    """
    print(f"\n🌊 Creating 3D interaction plot: {feature1}, {feature2}, {feature3}")
    
    fig = go.Figure(data=[go.Scatter3d(
        x=df[feature1],
        y=df[feature2],
        z=df[feature3],
        mode='markers',
        marker=dict(
            size=3,
            color=df[feature3],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title=feature3),
            opacity=0.7
        ),
        hovertemplate=f'<b>{feature1}:</b> %{{x:.4f}}<br>' +
                      f'<b>{feature2}:</b> %{{y:.4f}}<br>' +
                      f'<b>{feature3}:</b> %{{z:.4f}}<extra></extra>'
    )])
    
    fig.update_layout(
        title=dict(
            text=f'<b>3D Feature Interaction: {feature1} vs {feature2} vs {feature3}</b>',
            x=0.5,
            font=dict(size=16)
        ),
        scene=dict(
            xaxis_title=feature1,
            yaxis_title=feature2,
            zaxis_title=feature3,
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        height=700,
        width=1000,
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved 3D interaction plot to {save_path}")
    
    fig.write_html(HTML_PATH / f'3d_interaction_{feature1}_{feature2}.html')
    fig.write_image(VIZ_PATH / f'3d_interaction_{feature1}_{feature2}.png', width=1000, height=700)
    
    return fig


def plot_feature_correlations_with_target(df, target_col='target', save_path=None):
    """
    Create correlation bar chart with target variable
    
    Parameters:
    - df: DataFrame with features and target
    - target_col: name of target column
    - save_path: path to save the plot
    """
    print(f"\n📊 Creating feature correlation with target: {target_col}")
    
    # Calculate correlations with target
    correlations = df.corr()[target_col].drop(target_col).sort_values()
    
    fig = go.Figure(data=go.Bar(
        x=correlations.values,
        y=correlations.index,
        orientation='h',
        marker=dict(
            color=correlations.values,
            colorscale='RdYlBu_r',
            showscale=True,
            colorbar=dict(title='Correlation', x=1.02)
        ),
        text=[f'{val:.3f}' for val in correlations.values],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Correlation: %{x:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'<b>Feature Correlations with {target_col}</b><br><sub>Absolute correlation ranking</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis=dict(title='Correlation Coefficient', range=[-1, 1]),
        yaxis=dict(title='Features', autorange='reversed'),
        height=500,
        width=900,
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved correlation chart to {save_path}")
    
    fig.write_html(HTML_PATH / 'target_correlations.html')
    fig.write_image(VIZ_PATH / 'target_correlations.png', width=900, height=500)
    
    return fig


def generate_all_distribution_plots(df, target_col='target', save_path=None):
    """
    Generate all feature distribution visualizations
    
    Parameters:
    - df: DataFrame with features and target
    - target_col: name of target column
    - save_path: output directory (optional)
    """
    print("\n" + "="*60)
    print("🎨 Generating Complete Feature Distribution Analysis")
    print("="*60)
    
    # 1. Feature distributions
    plot_animated_feature_distributions(
        df, 
        save_path=HTML_PATH / 'feature_distributions_complete.html'
    )
    
    # 2. Target distribution (if target exists)
    if target_col in df.columns:
        plot_animated_target_distribution(
            df[target_col].values,
            df[target_col].values,  # Using same for demo
            save_path=HTML_PATH / 'target_analysis.html'
        )
    
    # 3. Correlation matrix
    plot_animated_correlation_matrix(
        df,
        save_path=HTML_PATH / 'correlation_analysis.html'
    )
    
    # 4. Violin distributions
    plot_animated_violin_distributions(
        df,
        save_path=HTML_PATH / 'violin_analysis.html'
    )
    
    # 5. Correlations with target
    if target_col in df.columns:
        plot_feature_correlations_with_target(
            df, target_col,
            save_path=HTML_PATH / 'target_correlations_analysis.html'
        )
    
    print("\n" + "="*60)
    print("✅ All distribution visualizations generated successfully!")
    print(f"📁 Output directory: {HTML_PATH}")
    print("="*60)


if __name__ == "__main__":
    # Demo with synthetic data
    np.random.seed(42)
    n_samples = 1000
    
    # Create synthetic features with different distributions
    df_demo = pd.DataFrame({
        'returns': np.random.normal(0.001, 0.02, n_samples),
        'volatility': np.random.gamma(2, 0.05, n_samples),
        'volume': np.random.lognormal(20, 1, n_samples),
        'rsi': np.random.uniform(30, 70, n_samples),
        'target': np.random.normal(0.001, 0.02, n_samples),
        'feature1': np.random.normal(0, 1, n_samples),
        'feature2': np.random.exponential(1, n_samples),
        'feature3': np.random.uniform(-2, 2, n_samples)
    })
    
    # Generate all visualizations
    generate_all_distribution_plots(df_demo, target_col='target')
    
    # Create 3D interaction plot example
    plot_3d_feature_interaction(df_demo, 'returns', 'volatility', 'rsi')


🎨 Generating Complete Feature Distribution Analysis

📊 Creating animated feature distribution dashboard...
✅ Saved feature distributions to visualizations\features\html\feature_distributions_complete.html


Resorting to unclean kill browser.



📊 Feature Statistics Summary:
   returns: μ=0.0014, med=0.0015, skew=0.117, kurt=0.066
   volatility: μ=0.1024, med=0.0866, skew=1.231, kurt=1.877
   volume: μ=812365378.0025, med=477144602.0691, skew=9.284, kurt=152.415
   rsi: μ=49.8874, med=49.3838, skew=0.044, kurt=-1.191
   target: μ=0.0006, med=0.0006, skew=0.023, kurt=0.210
   feature1: μ=-0.0389, med=-0.0536, skew=0.056, kurt=-0.111
   feature2: μ=0.9614, med=0.6796, skew=1.954, kurt=5.283
   feature3: μ=-0.0264, med=-0.0269, skew=0.024, kurt=-1.188

📊 Creating animated target distribution dashboard...
✅ Saved target distribution to visualizations\features\html\target_analysis.html

📊 Target Statistics:
   Train: μ=0.0006, σ=0.0198, skew=0.023, kurt=0.210
   Test:  μ=0.0006, σ=0.0198, skew=0.023, kurt=0.210

📊 Creating animated correlation matrix...
✅ Saved correlation matrix to visualizations\features\html\correlation_analysis.html


Resorting to unclean kill browser.



📊 High Correlation Pairs (>0.8 or <-0.8):

📊 Creating animated violin distribution dashboard...
✅ Saved violin distributions to visualizations\features\html\violin_analysis.html


Resorting to unclean kill browser.



📊 Creating feature correlation with target: target
✅ Saved correlation chart to visualizations\features\html\target_correlations_analysis.html

✅ All distribution visualizations generated successfully!
📁 Output directory: visualizations\features\html

🌊 Creating 3D interaction plot: returns, volatility, rsi


Resorting to unclean kill browser.
